In [3]:
api_keys = [

]


In [4]:
import re
import requests

# Color codes for clean terminal output
GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
RESET = "\033[0m"

def validate_maps_js_key(api_key):
    """
    Validates a single Google API Key specifically for the Maps JavaScript API.
    """
    # The actual authentication endpoint hit by the client-side JS library
    url = f"https://maps.googleapis.com/maps/api/js/AuthenticationService.Authenticate?1shttp://localhost&key={api_key}&callback=_"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "http://localhost/"  # Simulating a local web server execution context
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            content = response.text

            # Scenario 1: Key is broken/completely invalid
            if "InvalidKeyMapError" in content:
                return "INVALID", "The API key is completely invalid or does not exist."

            # Scenario 2: Key is valid but the specific Maps JS API is disabled
            elif "ApiNotActivatedMapError" in content:
                return "NOT_ACTIVATED", "Key exists, but 'Maps JavaScript API' is not enabled in the Cloud Console."

            # Scenario 3: General API Error parsing
            elif "Quota My Project" in content or "error" in content.lower():
                # Extract specific error string if visible in response format
                error_match = re.search(r'\["(.*?)Error"', content)
                error_name = error_match.group(1) if error_match else "Unknown Error"
                return "RESTRICTED/ERROR", f"Failed with {error_name} (likely Quota or Restrictions)."

            # Scenario 4: Successful initialization handshake pattern match
            else:
                return "VALID", "Key is active and configured for Maps JavaScript API."

        else:
            return "ERROR", f"HTTP Status {response.status_code} received."

    except requests.exceptions.RequestException as e:
        return "CONNECTION_FAILURE", str(e)


def main():

    print("\n=== Starting Google Maps JS API Key Validation ===\n")

    for idx, key in enumerate(api_keys, 1):
        print(f"[{idx}/{len(api_keys)}] Checking: {key[:12]}...{key[-4:]}")
        status, reason = validate_maps_js_key(key.strip())

        if status == "VALID":
            print(f" -> Status: {GREEN}✅ {status}{RESET}")
        elif status == "NOT_ACTIVATED":
            print(f" -> Status: {YELLOW}⚠️ {status}{RESET} ({reason})")
        else:
            print(f" -> Status: {RED}❌ {status}{RESET} ({reason})")
        print("-" * 50)

if __name__ == "__main__":
    main()


=== Starting Google Maps JS API Key Validation ===

[1/129] Checking: AIzaSyA-2N9f...3QTw
 -> Status: ❌ RESTRICTED/ERROR (Failed with Unknown Error (likely Quota or Restrictions).)
--------------------------------------------------
[2/129] Checking: AIzaSyA-kcU5...hKco
 -> Status: ❌ RESTRICTED/ERROR (Failed with Unknown Error (likely Quota or Restrictions).)
--------------------------------------------------
[3/129] Checking: AIzaSyA-RG4h...Zf6w
 -> Status: ❌ RESTRICTED/ERROR (Failed with Unknown Error (likely Quota or Restrictions).)
--------------------------------------------------
[4/129] Checking: AIzaSyA0_bXg...NagY
 -> Status: ❌ RESTRICTED/ERROR (Failed with Unknown Error (likely Quota or Restrictions).)
--------------------------------------------------
[5/129] Checking: AIzaSyA0i5mA...AJfg
 -> Status: ❌ RESTRICTED/ERROR (Failed with Unknown Error (likely Quota or Restrictions).)
--------------------------------------------------
[6/129] Checking: AIzaSyA0WCH9...yqZs
 -> Statu